# SpineNET — Architecture Exploration

Sandbox notebook: compare keypoint detection vs segmentation
architectures on MaIA Scoliosis Dataset.

- **Resolution:** 512 x 256 (H x W)
- **Compute:** device-agnostic via `ai.utils.get_device()`
- **Cobb validation:** post-hoc via `ai.evaluation.cobb`
- **Runs logged to:** in-memory `RUN_LOG` list

In [ ]:
import json
import math
import os
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

_nb_dir = Path.cwd()
for _p in [_nb_dir, *_nb_dir.parents]:
    if (_p / "ai").is_dir() and (_p / "params.yaml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate repo root")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT = {REPO_ROOT}")

In [ ]:
with open(REPO_ROOT / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

SEED: int = int(PARAMS["data"]["random_seed"])
TEST_SIZE: float = float(PARAMS["data"]["test_size"])
BATCH_SIZE: int = int(PARAMS["train"]["batch_size"])
LEARNING_RATE: float = float(PARAMS["train"]["learning_rate"])
EPOCHS: int = int(PARAMS["train"]["epochs"])

DATA_ROOT = REPO_ROOT / "data" / "raw" / "MaIA_Scoliosis_Dataset"
AUDIT_DIR = REPO_ROOT / "data" / "processed" / "audit"
CLEAN_INDEX_PATH = AUDIT_DIR / "clean_index.csv"
CHECKPOINT_DIR = REPO_ROOT / "data" / "processed" / "spinenet_ckpts"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMG_H, IMG_W = 512, 256
NUM_TARGETS = 17           # T1..L5
NUM_KPS = NUM_TARGETS * 4  # 68
NUM_SEG_CLASSES = 1 + NUM_TARGETS  # 18

print(f"SEED={SEED}  BS={BATCH_SIZE}  LR={LEARNING_RATE}  EPOCHS={EPOCHS}")

In [ ]:
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

from ai.utils import get_device, describe_device

DEVICE_INFO = get_device()
DEVICE = DEVICE_INFO.device
print(describe_device(DEVICE_INFO))

In [ ]:
from ai.evaluation.cobb import cobb_from_keypoints, cobb_from_segmentation
from ai.preprocessing.keypoints import (
    KEYPOINTS_PER_VERTEBRA,
    TOTAL_KEYPOINTS,
    multiclass_mask_to_keypoints,
)
from ai.preprocessing.segmentation import remap_to_target_classes

In [ ]:
def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def severity_bucket(deg: float) -> str:
    if deg < 10: return "normal"
    if deg < 25: return "mild"
    if deg < 40: return "moderate"
    return "severe"

## 1 · Data loading

In [ ]:
assert CLEAN_INDEX_PATH.exists(), f"Run the audit notebook first: {CLEAN_INDEX_PATH}"
CLEAN_INDEX = pd.read_csv(CLEAN_INDEX_PATH)

# Keep ok + warn only
TRAINABLE = CLEAN_INDEX[CLEAN_INDEX["status"].isin(["ok", "warn"])].copy()

# Hard drops (known bad cases)
HARD_DROPS = {"S_0107"}
TRAINABLE = TRAINABLE[~TRAINABLE["image_path"].apply(
    lambda p: Path(p).stem in HARD_DROPS
)].reset_index(drop=True)

print(f"clean_index: {len(CLEAN_INDEX)}  trainable: {len(TRAINABLE)}")
for cat, n in TRAINABLE["category"].value_counts().items():
    print(f"  {cat:<10} {int(n):>4}")

In [ ]:
def patient_stratified_split(
    df: pd.DataFrame,
    *,
    test_size: float = TEST_SIZE,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split by (patient_id, category), stratified by category."""
    rng = np.random.default_rng(seed)
    train_keys: list[tuple] = []
    val_keys: list[tuple] = []
    for category, group in df.groupby("category"):
        pids = sorted(group["patient_id"].unique(), key=str)
        perm = rng.permutation(len(pids))
        pids_shuf = [pids[i] for i in perm]
        n_val = max(1, int(round(len(pids_shuf) * test_size)))
        val_keys.extend((pid, category) for pid in pids_shuf[:n_val])
        train_keys.extend((pid, category) for pid in pids_shuf[n_val:])

    train_set, val_set = set(train_keys), set(val_keys)
    keys = list(zip(df["patient_id"], df["category"]))
    train_mask = np.array([k in train_set for k in keys])
    val_mask = np.array([k in val_set for k in keys])
    return df[train_mask].reset_index(drop=True), df[val_mask].reset_index(drop=True)


TRAIN_DF, VAL_DF = patient_stratified_split(TRAINABLE)
assert len(TRAIN_DF) + len(VAL_DF) == len(TRAINABLE)
print(f"train: {len(TRAIN_DF)}  val: {len(VAL_DF)}")

## 2 · Dataset + DataLoader

In [ ]:
def hflip_keypoints(kps: np.ndarray, img_w: int) -> np.ndarray:
    out = kps.copy()
    out[:, 0] = (img_w - 1) - out[:, 0]
    k = out.reshape(-1, KEYPOINTS_PER_VERTEBRA, 2)
    k[:, [0, 1]] = k[:, [1, 0]]
    k[:, [2, 3]] = k[:, [3, 2]]
    return k.reshape(-1, 2)


def affine_keypoints(
    kps: torch.Tensor, valid: torch.Tensor,
    angle_deg: float, tx: float, ty: float, scale: float,
    w: int, h: int,
) -> torch.Tensor:
    cx, cy = w / 2.0, h / 2.0
    a = math.radians(angle_deg)
    cos_a, sin_a = math.cos(a), math.sin(a)
    out = kps.clone()
    dx, dy = out[:, 0] - cx, out[:, 1] - cy
    out[:, 0] = scale * (cos_a * dx + sin_a * dy) + cx + tx
    out[:, 1] = scale * (-sin_a * dx + cos_a * dy) + cy + ty
    out[~valid] = 0.0
    return out

In [ ]:
class SpineDataset(Dataset):
    """Yields (image, targets, meta). Targets: keypoints, valid_kps, seg_mask."""

    def __init__(
        self, df: pd.DataFrame, *,
        img_h: int = IMG_H, img_w: int = IMG_W,
        augment: bool = False, seed: int | None = None,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.img_h, self.img_w = img_h, img_w
        self.augment = augment
        self._rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        # Load + resize image
        img_pil = Image.open(row["image_path"]).convert("L")
        orig_w, orig_h = img_pil.size
        img_np = np.asarray(
            img_pil.resize((self.img_w, self.img_h), Image.BILINEAR),
            dtype=np.float32,
        ) / 255.0

        # Load + remap + resize seg mask
        mask_np = np.array(Image.open(row["multiclass_mask_path"]))
        if mask_np.ndim == 3:
            mask_np = mask_np[..., 0]
        mask_np = mask_np.astype(np.int32)
        remapped = remap_to_target_classes(mask_np)
        seg_np = np.array(
            Image.fromarray(remapped, mode="L").resize(
                (self.img_w, self.img_h), Image.NEAREST
            ),
            dtype=np.int64,
        )

        # Keypoints: extract + scale to training resolution
        kps = multiclass_mask_to_keypoints(mask_np).astype(np.float32).copy()
        kps[:, 0] *= self.img_w / orig_w
        kps[:, 1] *= self.img_h / orig_h

        # Horizontal flip
        if self.augment and self._rng.random() < 0.5:
            img_np = img_np[:, ::-1].copy()
            seg_np = seg_np[:, ::-1].copy()
            kps = hflip_keypoints(kps, self.img_w)

        # Pack tensors
        image_t = torch.from_numpy(img_np).unsqueeze(0)
        seg_t = torch.from_numpy(seg_np)
        valid = np.isfinite(kps).all(axis=1)
        kps_filled = np.where(valid[:, None], kps, 0.0).astype(np.float32)
        kps_t = torch.from_numpy(kps_filled)
        valid_t = torch.from_numpy(valid)

        # Affine augmentation (tensor space)
        if self.augment and self._rng.random() < 0.5:
            angle = float(self._rng.uniform(-5, 5))
            tx = float(self._rng.uniform(-0.05, 0.05)) * self.img_w
            ty = float(self._rng.uniform(-0.05, 0.05)) * self.img_h
            sc = float(self._rng.uniform(0.9, 1.1))
            image_t = TF.affine(
                image_t, angle=angle, translate=[tx, ty], scale=sc,
                shear=[0.0], interpolation=TF.InterpolationMode.BILINEAR,
            )
            seg_3d = TF.affine(
                seg_t.unsqueeze(0).float(), angle=angle, translate=[tx, ty],
                scale=sc, shear=[0.0], interpolation=TF.InterpolationMode.NEAREST,
            )
            seg_t = seg_3d.squeeze(0).long()
            kps_t = affine_keypoints(
                kps_t, valid_t, angle, tx, ty, sc, self.img_w, self.img_h,
            )

        # Intensity jitter
        if self.augment and self._rng.random() < 0.5:
            gain = float(self._rng.uniform(0.8, 1.2))
            bias = float(self._rng.uniform(-0.1, 0.1))
            image_t = (image_t * gain + bias).clamp(0, 1)

        targets = {"keypoints": kps_t, "valid_kps": valid_t, "seg_mask": seg_t}
        meta = {
            "patient_id": row.get("patient_id"),
            "category": row.get("category"),
            "orig_size": (orig_h, orig_w),
        }
        return image_t, targets, meta

In [ ]:
def make_loaders(
    train_df: pd.DataFrame, val_df: pd.DataFrame, *,
    batch_size: int = BATCH_SIZE, seed: int = SEED,
) -> tuple[DataLoader, DataLoader]:
    train_ds = SpineDataset(train_df, augment=True, seed=seed)
    val_ds = SpineDataset(val_df, augment=False)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0),
    )


# Smoke test
_ds = SpineDataset(TRAIN_DF.head(1), augment=False)
_img, _tgt, _meta = _ds[0]
print(f"image: {_img.shape}  seg: {_tgt['seg_mask'].shape}  kps: {_tgt['keypoints'].shape}")
del _ds, _img, _tgt, _meta

## 3 · Model architectures

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetBackbone(nn.Module):
    """4-level U-Net encoder+decoder. Returns (feature_map, bottleneck)."""

    def __init__(self, in_ch: int = 1, init_ch: int = 64) -> None:
        super().__init__()
        c1, c2, c3, c4 = init_ch, init_ch * 2, init_ch * 4, init_ch * 8
        self.enc1 = DoubleConv(in_ch, c1)
        self.enc2 = DoubleConv(c1, c2)
        self.enc3 = DoubleConv(c2, c3)
        self.enc4 = DoubleConv(c3, c4)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(c4, c3, 2, stride=2)
        self.dec3 = DoubleConv(c4, c3)
        self.up2 = nn.ConvTranspose2d(c3, c2, 2, stride=2)
        self.dec2 = DoubleConv(c3, c2)
        self.up1 = nn.ConvTranspose2d(c2, c1, 2, stride=2)
        self.dec1 = DoubleConv(c2, c1)
        self.out_channels = c1
        self.bottleneck_channels = c4

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return d1, e4

In [ ]:
class KeypointRegressor(nn.Module):
    """Bottleneck -> global pool -> MLP -> (B, 68, 2)."""

    def __init__(self, init_ch: int = 64) -> None:
        super().__init__()
        self.backbone = UNetBackbone(in_ch=1, init_ch=init_ch)
        bot_c = self.backbone.bottleneck_channels
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(bot_c, 256), nn.ReLU(inplace=True),
            nn.Linear(256, TOTAL_KEYPOINTS * 2),
        )

    def forward(self, x):
        _, bot = self.backbone(x)
        return self.head(self.pool(bot)).view(-1, TOTAL_KEYPOINTS, 2)


class KeypointHeatmap(nn.Module):
    """Feature map -> 68 heatmap channels; argmax -> coords."""

    def __init__(self, init_ch: int = 64) -> None:
        super().__init__()
        self.backbone = UNetBackbone(in_ch=1, init_ch=init_ch)
        self.head = nn.Conv2d(self.backbone.out_channels, TOTAL_KEYPOINTS, 1)

    def forward(self, x):
        feat, _ = self.backbone(x)
        return self.head(feat)


class Segmenter(nn.Module):
    """Feature map -> 1x1 conv -> (B, C, H, W)."""

    def __init__(self, init_ch: int = 64) -> None:
        super().__init__()
        self.backbone = UNetBackbone(in_ch=1, init_ch=init_ch)
        self.head = nn.Conv2d(self.backbone.out_channels, NUM_SEG_CLASSES, 1)

    def forward(self, x):
        feat, _ = self.backbone(x)
        return self.head(feat)


class SegmenterDeep(nn.Module):
    """Segmenter + extra refine block + dropout."""

    def __init__(self, init_ch: int = 64) -> None:
        super().__init__()
        self.backbone = UNetBackbone(in_ch=1, init_ch=init_ch)
        c = self.backbone.out_channels
        self.refine = nn.Sequential(DoubleConv(c, c), nn.Dropout2d(0.1))
        self.head = nn.Conv2d(c, NUM_SEG_CLASSES, 1)

    def forward(self, x):
        feat, _ = self.backbone(x)
        return self.head(self.refine(feat))

In [ ]:
for name, cls in [
    ("KeypointRegressor", KeypointRegressor),
    ("KeypointHeatmap", KeypointHeatmap),
    ("Segmenter", Segmenter),
    ("SegmenterDeep", SegmenterDeep),
]:
    m = cls(init_ch=64)
    print(f"  {name:22s}  {param_count(m)/1e6:.2f}M")
    del m

## 4 · Loss functions + heatmap utilities

In [ ]:
# -- Keypoint losses --

def masked_smooth_l1(pred: torch.Tensor, target: torch.Tensor,
                     valid: torch.Tensor) -> torch.Tensor:
    """Smooth L1 over valid keypoints. pred/target: (B,K,2), valid: (B,K)."""
    mask = valid.unsqueeze(-1).expand_as(pred).float()
    diff = F.smooth_l1_loss(pred * mask, target * mask, reduction="sum")
    return diff / mask.sum().clamp_min(1.0)


# -- Heatmap utilities --

def gaussian_heatmap(kps: torch.Tensor, valid: torch.Tensor,
                     h: int, w: int, sigma: float = 4.0) -> torch.Tensor:
    """Render (B, K, H, W) Gaussian heatmaps centred at each keypoint."""
    B, K, _ = kps.shape
    ys = torch.arange(h, device=kps.device).view(1, 1, h, 1).float()
    xs = torch.arange(w, device=kps.device).view(1, 1, 1, w).float()
    cx = kps[..., 0].view(B, K, 1, 1)
    cy = kps[..., 1].view(B, K, 1, 1)
    hm = torch.exp(-((xs - cx)**2 + (ys - cy)**2) / (2 * sigma**2))
    return hm * valid.view(B, K, 1, 1).float()


def coords_from_heatmap(hm: torch.Tensor) -> torch.Tensor:
    """(B, K, H, W) -> (B, K, 2) via argmax."""
    B, K, H, W = hm.shape
    idx = hm.view(B, K, -1).argmax(dim=-1)
    return torch.stack([(idx % W).float(), (idx // W).float()], dim=-1)


def masked_heatmap_mse(pred: torch.Tensor, target: torch.Tensor,
                       valid: torch.Tensor) -> torch.Tensor:
    """MSE over valid heatmap channels only."""
    per_ch = ((pred - target)**2).sum(dim=(-2, -1))
    valid_f = valid.float()
    return (per_ch * valid_f).sum() / valid_f.sum().clamp_min(1.0)


# -- Segmentation losses --

def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor,
                   num_classes: int = NUM_SEG_CLASSES,
                   eps: float = 1e-6) -> torch.Tensor:
    """1 - mean Dice over foreground classes (skip bg=0)."""
    probs = F.softmax(logits, dim=1)
    onehot = torch.zeros_like(probs)
    onehot.scatter_(1, target.unsqueeze(1), 1.0)
    probs, onehot = probs[:, 1:], onehot[:, 1:]
    dims = (0, 2, 3)
    inter = (probs * onehot).sum(dim=dims)
    card = probs.sum(dim=dims) + onehot.sum(dim=dims)
    return 1.0 - ((2.0 * inter + eps) / (card + eps)).mean()


SEG_CLASS_WEIGHTS = torch.tensor(
    [0.1] + [1.0] * NUM_TARGETS, device=DEVICE,
)


def seg_combo_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """0.3 * weighted CE + 0.7 * soft Dice."""
    ce = F.cross_entropy(logits, target, weight=SEG_CLASS_WEIGHTS)
    dice = soft_dice_loss(logits, target)
    return 0.3 * ce + 0.7 * dice

## 5 · Metrics + evaluation helpers

In [ ]:
def pck(pred: torch.Tensor, target: torch.Tensor,
        valid: torch.Tensor, img_h: int = IMG_H,
        img_w: int = IMG_W, thresh: float = 0.05) -> float:
    """Fraction of valid keypoints within `thresh` of image diagonal."""
    diag = (img_h**2 + img_w**2)**0.5
    dist = torch.linalg.norm(pred - target, dim=-1)
    hit = ((dist <= thresh * diag) & valid).float().sum()
    return float(hit / valid.float().sum().clamp_min(1.0))


def mean_dice(pred_idx: torch.Tensor, target_idx: torch.Tensor) -> float:
    """Mean Dice over classes 1..NUM_TARGETS (skip bg)."""
    dices: list[float] = []
    for cls in range(1, NUM_TARGETS + 1):
        p, t = (pred_idx == cls), (target_idx == cls)
        denom = p.float().sum().item() + t.float().sum().item()
        if denom == 0:
            continue
        dices.append(2.0 * (p & t).float().sum().item() / denom)
    return float(np.mean(dices)) if dices else 0.0

In [ ]:
def load_gt_cobb_cache() -> dict[str, float]:
    """Load all GT Cobb angles once. Key = 'S_{patient_id}'."""
    metrics_root = DATA_ROOT / "RadiographMetrics" / "metrics_json"
    cache: dict[str, float] = {}
    if not metrics_root.exists():
        return cache
    for p in metrics_root.glob("metrics_*.json"):
        try:
            with open(p) as f:
                data = json.load(f)
            pid = p.stem.replace("metrics_", "")
            cache[f"S_{pid}"] = float(data["cobb_angle_deg"])
        except (KeyError, ValueError):
            pass
    return cache


GT_COBB = load_gt_cobb_cache()
print(f"GT Cobb cache: {len(GT_COBB)} scoliosis cases")


def case_id(meta: dict) -> str:
    return f"{meta.get('category', '?')[0]}_{meta.get('patient_id', '?')}"


def gt_cobb_for(meta: dict) -> float:
    return GT_COBB.get(case_id(meta), float("nan"))

## 6 · Training engine

One generic `train()` function. Task-specific behavior injected via:
- `loss_fn(model, batch) -> loss` — computes the loss
- `eval_fn(model, batch) -> dict` — computes per-batch eval metrics

In [ ]:
@dataclass
class TrainResult:
    name: str
    task: str
    architecture: str
    history: list[dict] = field(default_factory=list)
    best_metrics: dict = field(default_factory=dict)
    hparams: dict = field(default_factory=dict)
    wall_clock_s: float = 0.0
    checkpoint_path: Path | None = None


RUN_LOG: list[TrainResult] = []


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    loss_fn,
    eval_fn,
    name: str,
    task: str,
    architecture: str,
    hparams: dict,
    epochs: int = EPOCHS,
    lr: float = LEARNING_RATE,
) -> TrainResult:
    """Generic training loop. Returns TrainResult appended to RUN_LOG."""
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optim, mode="min", factor=0.5, patience=5,
    )
    best_primary = -1.0
    best_state: dict | None = None
    best_metrics: dict = {}
    history: list[dict] = []

    t_total = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # Train
        model.train()
        tloss, n = 0.0, 0
        for batch in train_loader:
            loss = loss_fn(model, batch)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            tloss += float(loss); n += 1
        tloss /= max(n, 1)

        # Val
        model.eval()
        vloss, vn = 0.0, 0
        epoch_metrics: dict[str, list[float]] = {}
        with torch.no_grad():
            for batch in val_loader:
                vloss += float(loss_fn(model, batch)); vn += 1
                for k, v in eval_fn(model, batch).items():
                    epoch_metrics.setdefault(k, []).append(v)
        vloss /= max(vn, 1)
        sched.step(vloss)

        avg_metrics = {k: float(np.mean(vs)) for k, vs in epoch_metrics.items()}
        cur_lr = float(optim.param_groups[0]["lr"])
        row = {"epoch": epoch, "train_loss": tloss, "val_loss": vloss,
               "lr": cur_lr, "sec": time.time() - t0, **avg_metrics}
        history.append(row)

        # Track best by first metric in eval_fn output
        primary_key = next(iter(epoch_metrics))
        primary_val = avg_metrics[primary_key]
        if primary_val > best_primary:
            best_primary = primary_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = {**avg_metrics, "val_loss": vloss}

        # Print
        met_str = "  ".join(f"{k}={v:.3f}" for k, v in avg_metrics.items())
        print(f"  ep{epoch:02d}/{epochs}  tr={tloss:.4f}  va={vloss:.4f}  "
              f"{met_str}  lr={cur_lr:.1e}  ({time.time() - t0:.1f}s)")

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  restored best ({primary_key}={best_primary:.3f})")

    wall = time.time() - t_total
    result = TrainResult(
        name=name, task=task, architecture=architecture,
        history=history, best_metrics=best_metrics,
        hparams=hparams, wall_clock_s=wall,
    )
    RUN_LOG.append(result)
    return result

In [ ]:
# -- Keypoint regression --

def kp_regress_loss(model: nn.Module, batch) -> torch.Tensor:
    imgs, tgts, _ = batch
    pred = model(imgs.to(DEVICE))
    return masked_smooth_l1(pred, tgts["keypoints"].to(DEVICE),
                            tgts["valid_kps"].to(DEVICE))


def kp_regress_eval(model: nn.Module, batch) -> dict[str, float]:
    imgs, tgts, metas = batch
    pred = model(imgs.to(DEVICE))
    kps_gt = tgts["keypoints"].to(DEVICE)
    valid = tgts["valid_kps"].to(DEVICE)
    pck_val = pck(pred, kps_gt, valid)

    pred_np = pred.detach().cpu().numpy()
    cobb_errs: list[float] = []
    sev_hits: list[float] = []
    for i, m in enumerate(metas):
        gt = gt_cobb_for(m)
        if math.isnan(gt):
            continue
        oh, ow = m["orig_size"]
        kps_orig = pred_np[i].copy()
        kps_orig[:, 0] *= ow / IMG_W
        kps_orig[:, 1] *= oh / IMG_H
        pc = float(cobb_from_keypoints(kps_orig))
        cobb_errs.append(abs(pc - gt))
        sev_hits.append(float(severity_bucket(pc) == severity_bucket(gt)))

    cobb_mae = float(np.mean(cobb_errs)) if cobb_errs else float("nan")
    sev_acc = float(np.mean(sev_hits)) if sev_hits else float("nan")
    return {"pck": pck_val, "cobb_mae": cobb_mae, "sev_acc": sev_acc}


# -- Keypoint heatmap --

def kp_heatmap_loss(model: nn.Module, batch) -> torch.Tensor:
    imgs, tgts, _ = batch
    kps = tgts["keypoints"].to(DEVICE)
    valid = tgts["valid_kps"].to(DEVICE)
    target_hm = gaussian_heatmap(kps, valid, IMG_H, IMG_W)
    pred_hm = model(imgs.to(DEVICE))
    return masked_heatmap_mse(pred_hm, target_hm, valid)


def kp_heatmap_eval(model: nn.Module, batch) -> dict[str, float]:
    imgs, tgts, metas = batch
    kps_gt = tgts["keypoints"].to(DEVICE)
    valid = tgts["valid_kps"].to(DEVICE)
    pred_hm = model(imgs.to(DEVICE))
    pred_coords = coords_from_heatmap(pred_hm)
    pck_val = pck(pred_coords, kps_gt, valid)

    pred_np = pred_coords.detach().cpu().numpy()
    cobb_errs: list[float] = []
    sev_hits: list[float] = []
    for i, m in enumerate(metas):
        gt = gt_cobb_for(m)
        if math.isnan(gt):
            continue
        oh, ow = m["orig_size"]
        kps_orig = pred_np[i].copy()
        kps_orig[:, 0] *= ow / IMG_W
        kps_orig[:, 1] *= oh / IMG_H
        pc = float(cobb_from_keypoints(kps_orig))
        cobb_errs.append(abs(pc - gt))
        sev_hits.append(float(severity_bucket(pc) == severity_bucket(gt)))

    cobb_mae = float(np.mean(cobb_errs)) if cobb_errs else float("nan")
    sev_acc = float(np.mean(sev_hits)) if sev_hits else float("nan")
    return {"pck": pck_val, "cobb_mae": cobb_mae, "sev_acc": sev_acc}


# -- Segmentation --

def seg_loss(model: nn.Module, batch) -> torch.Tensor:
    imgs, tgts, _ = batch
    logits = model(imgs.to(DEVICE))
    return seg_combo_loss(logits, tgts["seg_mask"].to(DEVICE).long())


def seg_eval(model: nn.Module, batch) -> dict[str, float]:
    imgs, tgts, metas = batch
    logits = model(imgs.to(DEVICE))
    pred_idx = logits.argmax(dim=1)
    seg_gt = tgts["seg_mask"].to(DEVICE).long()
    dice_val = mean_dice(pred_idx, seg_gt)

    pred_np = pred_idx.detach().cpu().numpy()
    cobb_errs: list[float] = []
    sev_hits: list[float] = []
    for i, m in enumerate(metas):
        gt = gt_cobb_for(m)
        if math.isnan(gt):
            continue
        pc = float(cobb_from_segmentation(pred_np[i]))
        cobb_errs.append(abs(pc - gt))
        sev_hits.append(float(severity_bucket(pc) == severity_bucket(gt)))

    cobb_mae = float(np.mean(cobb_errs)) if cobb_errs else float("nan")
    sev_acc = float(np.mean(sev_hits)) if sev_hits else float("nan")
    return {"dice": dice_val, "cobb_mae": cobb_mae, "sev_acc": sev_acc}

## 7 · Plotting helpers

In [ ]:
def plot_history(result: TrainResult) -> None:
    """Loss curves + primary metric from a TrainResult."""
    df = pd.DataFrame(result.history)
    metric_cols = [c for c in df.columns
                   if c not in {"epoch", "train_loss", "val_loss", "lr", "sec"}]
    primary = metric_cols[0] if metric_cols else None

    ncols = 2 if primary else 1
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 3.5))
    if ncols == 1:
        axes = [axes]

    axes[0].plot(df["epoch"], df["train_loss"], label="train")
    axes[0].plot(df["epoch"], df["val_loss"], "--", label="val")
    axes[0].set(xlabel="epoch", ylabel="loss", title=f"{result.name} — loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if primary:
        axes[1].plot(df["epoch"], df[primary], color="green")
        axes[1].set(xlabel="epoch", ylabel=primary,
                    title=f"{result.name} — {primary}")
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_seg_predictions(model: nn.Module, val_df: pd.DataFrame,
                         n: int = 4) -> None:
    """3-panel grid: radiograph | GT seg | predicted seg."""
    ds = SpineDataset(val_df.head(n), augment=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[None, :]

    model.eval()
    for i in range(min(n, len(ds))):
        img_t, tgts, meta = ds[i]
        seg_gt = tgts["seg_mask"].numpy()
        with torch.no_grad():
            logits = model(img_t.unsqueeze(0).to(DEVICE))
        seg_pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        cid = case_id(meta)
        axes[i, 0].imshow(img_t.squeeze(0), cmap="gray")
        axes[i, 0].set_title(f"{cid}"); axes[i, 0].set_axis_off()
        axes[i, 1].imshow(seg_gt, cmap="tab20", vmin=0, vmax=NUM_SEG_CLASSES - 1)
        axes[i, 1].set_title("ground truth"); axes[i, 1].set_axis_off()
        axes[i, 2].imshow(seg_pred, cmap="tab20", vmin=0, vmax=NUM_SEG_CLASSES - 1)
        axes[i, 2].set_title("predicted"); axes[i, 2].set_axis_off()

    plt.tight_layout()
    plt.show()


def plot_kp_predictions(model: nn.Module, val_df: pd.DataFrame,
                        n: int = 4, is_heatmap: bool = False) -> None:
    """Overlay predicted keypoints (red) and GT (green) on radiographs."""
    ds = SpineDataset(val_df.head(n), augment=False)
    fig, axes = plt.subplots(n, 1, figsize=(6, 4 * n))
    if n == 1:
        axes = [axes]

    model.eval()
    for i in range(min(n, len(ds))):
        img_t, tgts, meta = ds[i]
        kps_gt = tgts["keypoints"].numpy()
        valid = tgts["valid_kps"].numpy()

        with torch.no_grad():
            inp = img_t.unsqueeze(0).to(DEVICE)
            if is_heatmap:
                pred_hm = model(inp)
                pred_kps = coords_from_heatmap(pred_hm).squeeze(0).cpu().numpy()
            else:
                pred_kps = model(inp).squeeze(0).cpu().numpy()

        ax = axes[i]
        ax.imshow(img_t.squeeze(0), cmap="gray")
        ax.scatter(kps_gt[valid, 0], kps_gt[valid, 1], s=8, c="lime",
                   label="GT", zorder=3)
        ax.scatter(pred_kps[valid, 0], pred_kps[valid, 1], s=8, c="red",
                   marker="x", label="pred", zorder=3)
        ax.set_title(case_id(meta))
        ax.set_axis_off()
        if i == 0:
            ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 8 · Keypoint experiments

In [ ]:
train_loader, val_loader = make_loaders(TRAIN_DF, VAL_DF)

kp_model = KeypointRegressor(init_ch=64).to(DEVICE)
kp_baseline = train(
    kp_model, train_loader, val_loader,
    loss_fn=kp_regress_loss, eval_fn=kp_regress_eval,
    name="kp-regressor-baseline", task="keypoint",
    architecture="regressor", hparams={"init_ch": 64},
    epochs=5, lr=LEARNING_RATE,
)
plot_history(kp_baseline)
plot_kp_predictions(kp_model, VAL_DF, n=4)

In [ ]:
kp_hm_model = KeypointHeatmap(init_ch=64).to(DEVICE)
kp_heatmap = train(
    kp_hm_model, train_loader, val_loader,
    loss_fn=kp_heatmap_loss, eval_fn=kp_heatmap_eval,
    name="kp-heatmap-baseline", task="keypoint",
    architecture="heatmap", hparams={"init_ch": 64},
    epochs=5, lr=LEARNING_RATE,
)
plot_history(kp_heatmap)
plot_kp_predictions(kp_hm_model, VAL_DF, n=4, is_heatmap=True)

In [ ]:
KP_SWEEP = [
    {"lr": 1e-3, "epochs": 20},
    {"lr": 5e-4, "epochs": 20},
    {"lr": 1e-4, "epochs": 20},
]

for cfg in KP_SWEEP:
    m = KeypointRegressor(init_ch=64).to(DEVICE)
    r = train(
        m, train_loader, val_loader,
        loss_fn=kp_regress_loss, eval_fn=kp_regress_eval,
        name=f"kp-reg-lr{cfg['lr']}", task="keypoint",
        architecture="regressor", hparams={"init_ch": 64, **cfg},
        epochs=cfg["epochs"], lr=cfg["lr"],
    )
    plot_history(r)

## 9 · Segmentation experiments

In [ ]:
seg_model = Segmenter(init_ch=64).to(DEVICE)
seg_baseline = train(
    seg_model, train_loader, val_loader,
    loss_fn=seg_loss, eval_fn=seg_eval,
    name="seg-baseline", task="segmentation",
    architecture="segmenter", hparams={"init_ch": 64},
    epochs=5, lr=LEARNING_RATE,
)
plot_history(seg_baseline)
plot_seg_predictions(seg_model, VAL_DF, n=4)

In [ ]:
seg_deep_model = SegmenterDeep(init_ch=64).to(DEVICE)
seg_deep = train(
    seg_deep_model, train_loader, val_loader,
    loss_fn=seg_loss, eval_fn=seg_eval,
    name="seg-deep", task="segmentation",
    architecture="segmenter_deep", hparams={"init_ch": 64},
    epochs=5, lr=LEARNING_RATE,
)
plot_history(seg_deep)
plot_seg_predictions(seg_deep_model, VAL_DF, n=4)

In [ ]:
SEG_SWEEP = [
    {"lr": 1e-3, "epochs": 20},
    {"lr": 5e-4, "epochs": 20},
    {"lr": 1e-4, "epochs": 20},
]

for cfg in SEG_SWEEP:
    m = Segmenter(init_ch=64).to(DEVICE)
    r = train(
        m, train_loader, val_loader,
        loss_fn=seg_loss, eval_fn=seg_eval,
        name=f"seg-lr{cfg['lr']}", task="segmentation",
        architecture="segmenter", hparams={"init_ch": 64, **cfg},
        epochs=cfg["epochs"], lr=cfg["lr"],
    )
    plot_history(r)

In [ ]:
plot_seg_predictions(m, VAL_DF, n=4)

## 10 · Run comparison

In [ ]:
def build_run_table() -> pd.DataFrame:
    if not RUN_LOG:
        print("No runs yet.")
        return pd.DataFrame()

    rows = []
    for r in RUN_LOG:
        m = r.best_metrics
        rows.append({
            "name": r.name,
            "task": r.task,
            "arch": r.architecture,
            "lr": r.hparams.get("lr", r.hparams.get("learning_rate", "")),
            "epochs": len(r.history),
            "val_loss": m.get("val_loss", float("nan")),
            "pck": m.get("pck", float("nan")),
            "dice": m.get("dice", float("nan")),
            "cobb_mae": m.get("cobb_mae", float("nan")),
            "sev_acc": m.get("sev_acc", float("nan")),
            "wall_s": r.wall_clock_s,
        })
    df = pd.DataFrame(rows)
    df = df.sort_values(["cobb_mae", "sev_acc"],
                        ascending=[True, False]).reset_index(drop=True)
    return df


run_df = build_run_table()
run_df

In [ ]:
if not run_df.empty:
    best = run_df.iloc[0]
    print(f"Best run: {best['name']}")
    print(f"  cobb_mae = {best['cobb_mae']:.2f}")
    print(f"  sev_acc  = {best['sev_acc']:.3f}")
    print(f"  wall     = {best['wall_s']:.0f}s")